In [ ]:
Twi_API=""

In [ ]:
# @title
import tweepy
import pandas as pd

# 1. Twitter API credentials (replace with your own)
bearer_token = Twi_API

# 2. Authenticate with the API
client = tweepy.Client(bearer_token=bearer_token, wait_on_rate_limit=True)

# 3. Movies list
movies = [
    {"title": "Atypical", "type": "tv", "year": 2017},
    {"title": "The Good Doctor", "type": "tv", "year": 2017},
]

# 4. Function to collect tweets
def get_tweets_for_movie(movie_title, max_tweets=100):
    query = f'"{movie_title}" -is:retweet lang:en'
    tweets = []
    for tweet in tweepy.Paginator(client.search_recent_tweets,
                                  query=query,
                                  tweet_fields=['created_at','text','author_id'],
                                  max_results=100).flatten(limit=max_tweets):
        tweets.append({
            "movie": movie_title,
            "author_id": tweet.author_id,
            "created_at": tweet.created_at,
            "text": tweet.text
        })
    return tweets

# 5. Collect tweets for all movies
all_tweets = []
for movie in movies:
    print(f"Collecting tweets for {movie['title']}...")
    all_tweets.extend(get_tweets_for_movie(movie['title'], max_tweets=200))

# 6. Convert to DataFrame
df = pd.DataFrame(all_tweets)
print(df.head())

# 7. Save to CSV
df.to_csv("twitter_movie_reviews.csv", index=False)


In [ ]:
# @title
import requests
import pandas as pd
import re
import time

# --- CONFIG ---
bearer_token = Twi_API  # replace with your key
base_url = "https://api.twitter.com/2/tweets/search/recent"
pause_between_requests = 2  # free-tier safe pacing
max_results = 50  # max per request
# ----------------

# Movies/shows
movies = [
    {"title": "Atypical", "type": "tv", "year": 2017},
    {"title": "The Good Doctor", "type": "tv", "year": 2017}
    {"title": "Temple Grandin", "type": "film", "year": 2010},
    {"title": "Everything's Gonna Be Okay", "type": "tv", "year": 2020},
    {"title": "Taare Zameen Par", "type": "film", "year": 2007, "alt_titles": ["Like Stars on Earth"]},
    {"title": "Sitare Zameen Par", "type": "film", "year": 2025},
    {"title": "Front of the Class", "type": "film", "year": 2008},
    {"title": "The Tic Code", "type": "film", "year": 1998},
    {"title": "Patience", "type": "film", "year": 2017},
    {"title": "Barfi", "type": "film", "year": 2012},
    {"title": "My Name is Khan", "type": "film", "year": 2010, "alt_titles": ["MNIK"]},
    {"title": "Music", "type": "film", "year": 2021, "director": "Sia"},
    {"title": "Hichki", "type": "film", "year": 2018, "alt_titles": ["Hiccup"]},  # informal translation
    {"title": "Rain Man", "type": "film", "year": 1988},
    {"title": "Koi... Mil Gaya", "type": "film", "year": 2003, "alt_titles": ["Found Someone", "I Have Found Someone"]},
    {"title": "Extraordinary Attorney Woo", "type": "tv", "year": 2022, "alt_titles": ["Weird Lawyer Woo Young-woo"]}oo"]}
]

neuro_terms = ["autism", "autistic", "neurodivergent", "Asperger", "ADHD", "representation", "portrayal"]
context_words = ['film', 'movie', 'series', 'show', 'character', 'portrays',
                 'depicts', 'stars', 'performance', 'actor', 'actress',
                 'review', 'watch', 'episode', 'season']

# --- Helper functions ---

def mentions_title_with_boundary(text: str, title: str, alt_titles: list = None) -> bool:
    lower_text = text.lower()
    if len(title.split()) <= 2:
        pattern = r'\b' + re.escape(title.lower()) + r'\b'
        if re.search(pattern, lower_text):
            return True
    else:
        if title.lower() in lower_text:
            return True
    if alt_titles:
        for alt in alt_titles:
            if alt.lower() in lower_text:
                return True
    return False

def has_context_words(text: str) -> bool:
    lower_text = text.lower()
    return any(word in lower_text for word in context_words)

def count_mentions(text: str, title: str, alt_titles: list = None) -> int:
    lower_text = text.lower()
    count = 0
    if len(title.split()) <= 2:
        pattern = r'\b' + re.escape(title.lower()) + r'\b'
        count += len(re.findall(pattern, lower_text))
    else:
        start = 0
        while True:
            pos = lower_text.find(title.lower(), start)
            if pos == -1:
                break
            count += 1
            start = pos + 1
    if alt_titles:
        for alt in alt_titles:
            count += lower_text.count(alt.lower())
    return count


def search_tweets(movie_info: dict) -> list:
    title = movie_info["title"]
    alt_titles = movie_info.get("alt_titles", [])
    neuro_query = " OR ".join(neuro_terms)

    # Build Twitter search query (case-insensitive)
    query = f'("{title}" OR {" OR ".join([f"{alt}" for alt in alt_titles])}) ({neuro_query}) -is:retweet lang:en'

    headers = {"Authorization": f"Bearer {bearer_token}"}
    params = {
        "query": query,
        "max_results": max_results,
        "tweet.fields": "created_at,lang,author_id,public_metrics",
        "expansions": "author_id",
        "user.fields": "username,verified,public_metrics"
    }

    tweets = []
    seen_ids = set()

    print(f"Searching tweets for: {title} ...")
    try:
        resp = requests.get(base_url, headers=headers, params=params)
        if resp.status_code != 200:
            print(f"  Twitter API error {resp.status_code} for {title}: {resp.text}")
            return tweets

        data = resp.json()
        tweet_data = data.get("data", [])
        users = {u["id"]: u for u in data.get("includes", {}).get("users", [])}

        for t in tweet_data:
            if t["id"] in seen_ids:
                continue
            seen_ids.add(t["id"])

            text = t["text"]
            if not mentions_title_with_boundary(text, title, alt_titles):
                continue
            if not has_context_words(text):
                continue

            mention_count = count_mentions(text, title, alt_titles)
            author = users.get(t["author_id"], {})
            tweets.append({
                "tweet_id": t["id"],
                "movie_title": title,
                "movie_type": movie_info.get("type", "film"),
                "user": author.get("username", ""),
                "verified": author.get("verified", False),
                "followers": author.get("public_metrics", {}).get("followers_count", 0),
                "tweet_text": text,
                "created_at": t["created_at"],
                "like_count": t["public_metrics"]["like_count"],
                "retweet_count": t["public_metrics"]["retweet_count"],
                "reply_count": t["public_metrics"]["reply_count"],
                "mention_count": mention_count,
                "source": "Twitter"
            })

        print(f"  Found {len(tweets)} tweets for {title}")

    except Exception as e:
        print(f"  Error fetching tweets for {title}: {e}")

    time.sleep(pause_between_requests)
    return tweets


# --- Main Execution ---
print("Starting Twitter search...\n")
all_tweets = []

for movie in movies:
    tweets = search_tweets(movie)
    all_tweets.extend(tweets)

df_tweets = pd.DataFrame(all_tweets)

# --- Clean and filter ---
if not df_tweets.empty:
    df_tweets.drop_duplicates(subset=["tweet_text"], inplace=True)
    df_tweets = df_tweets.sort_values(["movie_title", "like_count"], ascending=[True, False])
    df_tweets.to_csv("twitter_neurodivergence_tweets.csv", index=False)
    print(f"\nTotal tweets found: {len(df_tweets)}")
    print(f"File saved: twitter_neurodivergence_tweets.csv")
    print(df_tweets.head(10))
else:
    print("\nNo tweets found. Try widening your search terms or check your API key.")


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time, random

# --- CONFIG ---
nitter_base = "https://nitter.lucasferguson.dev"  # fallback mirror
movies = [
    {"title": "Atypical", "type": "tv", "year": 2017},
    {"title": "The Good Doctor", "type": "tv", "year": 2017},
    {"title": "Temple Grandin", "type": "film", "year": 2010},
]
neuro_terms = ["autism", "autistic", "ADHD", "neurodivergent", "Asperger"]
max_tweets_per_movie = 50

# --- Scraper ---
def search_tweets_nitter(query, max_results=50):
    results = []
    url = f"{nitter_base}/search?f=tweets&q={requests.utils.quote(query)}"
    headers = {"User-Agent": "Mozilla/5.0"}
    try:
        resp = requests.get(url, headers=headers, timeout=10)
        if resp.status_code != 200:
            print(f"⚠️ Nitter error {resp.status_code} for query: {query}")
            return results

        soup = BeautifulSoup(resp.text, "html.parser")
        tweets = soup.select(".timeline-item")

        for tweet in tweets[:max_results]:
            content_tag = tweet.select_one(".tweet-content")
            if not content_tag:
                continue
            text = content_tag.get_text(" ", strip=True)
            username = tweet.select_one(".username").get_text(strip=True) if tweet.select_one(".username") else None
            date_tag = tweet.select_one("a.tweet-date")
            date = date_tag["title"] if date_tag and date_tag.has_attr("title") else None
            results.append({"username": username, "text": text, "date": date})
    except Exception as e:
        print(f"❌ Error fetching {query}: {e}")
    return results


# --- Main Loop ---
all_tweets = []
print("Starting Nitter scrape...\n")

for movie in movies:
    title = movie["title"]
    for term in neuro_terms:
        query = f'"{title}" {term} since:2020-01-01 until:2025-12-31 lang:en'
        print(f"🔎 Searching: {query}")
        tweets = search_tweets_nitter(query, max_tweets_per_movie)
        for t in tweets:
            t.update({
                "movie_title": title,
                "movie_type": movie.get("type", ""),
                "term": term,
                "source": "Nitter"
            })
        all_tweets.extend(tweets)
        time.sleep(random.uniform(1, 2))

df_twitter = pd.DataFrame(all_tweets)
df_twitter.to_csv("twitter_nitter_neurodivergence.csv", index=False)
print(f"\n✅ Done! Collected {len(df_twitter)} tweets (saved to twitter_nitter_neurodivergence.csv)")


Starting Nitter scrape...

🔎 Searching: "Atypical" autism since:2020-01-01 until:2025-12-31 lang:en
❌ Error fetching "Atypical" autism since:2020-01-01 until:2025-12-31 lang:en: HTTPSConnectionPool(host='nitter.lucasferguson.dev', port=443): Max retries exceeded with url: /search?f=tweets&q=%22Atypical%22%20autism%20since%3A2020-01-01%20until%3A2025-12-31%20lang%3Aen (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7e1e028f9a00>: Failed to resolve 'nitter.lucasferguson.dev' ([Errno -2] Name or service not known)"))
🔎 Searching: "Atypical" autistic since:2020-01-01 until:2025-12-31 lang:en
❌ Error fetching "Atypical" autistic since:2020-01-01 until:2025-12-31 lang:en: HTTPSConnectionPool(host='nitter.lucasferguson.dev', port=443): Max retries exceeded with url: /search?f=tweets&q=%22Atypical%22%20autistic%20since%3A2020-01-01%20until%3A2025-12-31%20lang%3Aen (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7e1e00639490>: Fail

In [ ]:
from google.colab import files

files.download("twitter_neurodivergence_tweets.csv")